In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_gtfs_shapes = spark.table("bg_traffic.bg_traffic_bronze.gtfs_shapes")
bronze_gtfs_shapes.display()

In [0]:
required_columns = {
    "shape_id","shape_pt_lat","shape_pt_lon","shape_pt_sequence","shape_dist_traveled"
}

missing_columns = required_columns - set(bronze_gtfs_shapes.columns)
if missing_columns:
    raise ValueError(
        "GRESKA: Izvorni GTFS shapes je promenio strukturu, postoje nedostajuce kolone!"
        )

In [0]:
bronze_gtfs_shapes.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_gtfs_shapes.columns]).show()

In [0]:
bronze_gtfs_shapes.printSchema()

### Casting

In [0]:
types_shapes = bronze_gtfs_shapes.select(
    F.col("shape_id").cast("string"),
    F.col("shape_pt_lat").cast("double"),
    F.col("shape_pt_lon").cast("double"),
    F.col("shape_pt_sequence").cast("integer"),
    F.col("shape_dist_traveled").cast("double")
)

types_shapes.printSchema()

### Dedup

In [0]:
dedup_shapes = types_shapes.dropDuplicates(["shape_id", "shape_pt_sequence"])

dedup_count = types_shapes.count() - types_shapes.count()
print(f"Broj duplikata: {dedup_count}")

### Valid

In [0]:
valid_shapes = dedup_shapes.filter(
    (F.col("shape_id").isNotNull()) &
    (F.col("shape_pt_sequence").isNotNull()) &
    (F.col("shape_pt_lat").between(44.0, 45.5)) &
    (F.col("shape_pt_lon").between(19.5, 21.0)) 
).withColumn("silver_processed_at",F.current_timestamp())

valid_shapes.display()

In [0]:
if valid_shapes.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

### Write in silver table

In [0]:
valid_shapes.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable("bg_traffic.bg_traffic_silver.gtfs_shape")